In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import gc
import numpy as np
from privacy_analysis.compute_privacy_sgm import compute_dp_sgd_privacy

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1), 
    transforms.Resize((32, 32)),               
    transforms.ToTensor(),                       
])

cifar_dataset = datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
train_loader_private = DataLoader(dataset=cifar_dataset, batch_size=64, shuffle=True)

cifar_test_dataset = datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)
test_loader = DataLoader(dataset=cifar_test_dataset, batch_size=1000, shuffle=False)

public_dataset = datasets.SVHN(root='./data', split='train', transform=transform, download=True)

np.random.seed(42)
public_indices = np.random.choice(len(public_dataset), size=2000, replace=False)
public_subset = Subset(public_dataset, public_indices)

train_loader_public = DataLoader(dataset=public_subset, batch_size=64, shuffle=True)

In [4]:
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.conv1 = nn.Conv2d(1,6, kernel_size=5, stride=1, padding=2)
        self.tanh1 = nn.Tanh()
        self.pool1 = nn.AvgPool2d(kernel_size=2,  stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5, stride=1)
        self.tanh2 = nn.Tanh()
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(16, 120, kernel_size=5, stride=1)
        self.tanh3 = nn.Tanh()
        self.fc1 = nn.Linear(480, 84)
        self.tanh4 = nn.Tanh()
        self.fc2 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool1(self.tanh1(self.conv1(x)))
        x = self.pool2(self.tanh2(self.conv2(x)))
        x = self.tanh3(self.conv3(x))
        x = x.view(-1, 480)
        x = self.tanh4(self.fc1(x))
        x = self.fc2(x)
        return x

model = Model()
criterion = nn.CrossEntropyLoss()
gc.collect()

487

In [5]:
def DOPESGD(model, D_loader, D_s_loader, test_loader, lr, sigma, C, l, T, delta):
    """
    Parameters:
        D_loader: private training data loader
        n: private batch size
        D_s_loader: public training data loader
        n_s: public batch size
        lr: learning rate
        sigma: noise scale
        C: gradient norm clip
        l: loss function
        T: number of training iterations
    """

    train_losses = []
    train_accuracies = []
    test_losses = []
    test_accuracies = []

    n_private = len(D_loader.dataset)
    batch_size_private = D_loader.batch_size
    
    optim = torch.optim.SGD(model.parameters(), lr)
    pub_iter = iter(D_s_loader)

    for epoch in range(T):
        
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for priv_inputs, priv_labels in D_loader:
            
            try:
                pub_inputs, pub_labels = next(pub_iter)
            except StopIteration:
                pub_iter = iter(D_s_loader)
                pub_inputs, pub_labels = next(pub_iter)
            
            optim.zero_grad()
            pub_out = model(pub_inputs)
            pub_loss = l(pub_out, pub_labels)
            pub_loss.backward()
            pub_grad = []
            for p in model.parameters():
                pub_grad.append(p.grad.detach().clone())
            pub_grad = [g.clone() for g in pub_grad]

            grads = [torch.zeros_like(p) for p in model.parameters()]

            for x, y in zip(priv_inputs, priv_labels):
                optim.zero_grad()
                output = model(x.unsqueeze(0))
                loss = l(output, y.unsqueeze(0))
                loss.backward()

                for i, p in enumerate(model.parameters()):
                    if p.requires_grad and p.grad is not None:
                        grads[i] += pub_grad[i] + ((p.grad - pub_grad[i]) * C) / (torch.max(torch.tensor(C), torch.norm(p.grad - pub_grad[i])))

                for i in range(len(grads)):    
                    grads[i] += torch.normal(0, sigma * C, grads[i].shape)

            for p, g in zip(model.parameters(), grads):
                if p.requires_grad:
                    p.grad = g / len(priv_inputs)
            optim.step()

            total_loss += pub_loss.item() * priv_inputs.size(0)
            with torch.no_grad():
                priv_out = model(priv_inputs)
                _, predicted = torch.max(priv_out.data, 1)
                total += priv_labels.size(0)
                correct += (predicted == priv_labels).sum().item()
        
        epoch_loss = total_loss / total
        epoch_accuracy = 100 * correct / total

        model.eval()
        correct = 0
        total = 0
        test_loss = 0.0

        with torch.no_grad():
            for inputs, labels in test_loader:
                outputs = model(inputs)
                loss = l(outputs, labels)
                test_loss += loss.item() * labels.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        test_accuracy = 100 * correct / total
        test_loss = test_loss / total

        train_losses.append(epoch_loss)
        train_accuracies.append(epoch_accuracy)
        test_losses.append(test_loss)
        test_accuracies.append(test_accuracy)

        eps, _ = compute_dp_sgd_privacy(n_private, batch_size_private, sigma, epoch + 1, delta)

        print(f"Epoch [{epoch+1}/{T}] ... Training Loss: {epoch_loss:.4f} ... Training Accuracy: {epoch_accuracy:.2f}% ... Test Loss: {test_loss:.4f} ... Test Accuracy: {test_accuracy:.2f}% ... Epsilon: {eps:.4f} ... Delta: {delta}")

    return train_losses, train_accuracies, test_losses, test_accuracies

In [6]:
train_losses, train_accuracies, test_losses, test_accuracies = DOPESGD(
                                                                        model=model,
                                                                        D_loader=train_loader_private,
                                                                        D_s_loader=train_loader_public,
                                                                        test_loader=test_loader,
                                                                        lr=0.01, 
                                                                        sigma=0.652, 
                                                                        C=1.0, 
                                                                        l=criterion,
                                                                        T=15,
                                                                        delta=1e-5
                                                                      )

DP-SGD with sampling rate = 0.128% and noise_multiplier = 0.652 iterated over 782 steps satisfies differential privacy with eps = 2.12 and delta = 1e-05.
The optimal RDP order is 5.5.
Epoch [1/15] ... Training Loss: 2.2842 ... Training Accuracy: 10.85% ... Test Loss: 2.2981 ... Test Accuracy: 10.23% ... Epsilon: 2.1246 ... Delta: 1e-05
DP-SGD with sampling rate = 0.128% and noise_multiplier = 0.652 iterated over 1563 steps satisfies differential privacy with eps = 2.27 and delta = 1e-05.
The optimal RDP order is 5.5.
Epoch [2/15] ... Training Loss: 2.2697 ... Training Accuracy: 13.72% ... Test Loss: 2.2502 ... Test Accuracy: 14.15% ... Epsilon: 2.2701 ... Delta: 1e-05
DP-SGD with sampling rate = 0.128% and noise_multiplier = 0.652 iterated over 2344 steps satisfies differential privacy with eps = 2.4 and delta = 1e-05.
The optimal RDP order is 5.0.
Epoch [3/15] ... Training Loss: 2.2422 ... Training Accuracy: 14.44% ... Test Loss: 2.2600 ... Test Accuracy: 12.12% ... Epsilon: 2.4016 ..